# 02 · Cleaning and feature engineering
**Renewable Energy Transition in France: Regional Dynamics and the Rise of Sustainable Electricity Production**

Author: Soumahoro Souleymane · EFREI Paris · Data Storytelling project (#EFREIDataStories2026)
Research question: *How are French regions progressing toward a renewable electricity future?*

In notebook 01 I audited `production-regionale-mensuelle-filiere.csv`. The table is clean and balanced (13 regions × 144 months, 2014-01 to 2025-12, no duplicates). The only point that needed a decision was the empty nuclear values.

In this notebook I turn the raw file into the tables the dashboard will use. I only clean and build features here; I do not explore the results yet (EDA comes next).

**What I produce**

| File | Content |
|---|---|
| `cleaned_dataset.csv` | one row per region and month, clean values and quality flags |
| `feature_engineered_dataset.csv` | the same rows with time features, shares, growth, rankings and scores |
| `dim_region.csv` | one row per region (names, centroid, nuclear flag, geometry check) |
| `fact_energy.csv` | long table: month × region × source |
| `dim_source.csv`, `annual_region_summary.csv`, `national_annual.csv` | small pre-aggregated tables for a fast app |
| `regions.geojson`, `feature_scales.json`, `feature_dictionary.csv` | map, score scaling and documentation |
| `data_quality_report.md`, `feature_engineering_report.md` | reports written from the numbers of this run |

The reusable code is in `utils/` (`config.py`, `prep.py`, `dictionary.py`) so the Streamlit app will use exactly the same cleaning and the same formulas.

## 0 · Setup
The notebook looks for the `utils/` folder above the notebook, so it runs from any clean checkout. The raw file is searched in `data/` first, then at my local path.

In [ ]:
import sys, os, json, importlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "utils" / "prep.py").exists():
            return p
    raise FileNotFoundError("Folder utils/ not found. Put it next to notebooks/ and data/.")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
from utils import config as cfg, prep, dictionary
for m in (cfg, prep, dictionary):
    importlib.reload(m)

candidates = [os.environ.get("DATA_CSV", ""), ROOT / "data" / cfg.RAW_FILE, cfg.LOCAL_RAW_PATH]
RAW_PATH = next((Path(p) for p in candidates if p and Path(p).exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError(f"{cfg.RAW_FILE} not found in {ROOT / 'data'}")

PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
PROC.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Raw file     :", RAW_PATH)
print("Processed to :", PROC)
print("Reports to   :", REPORTS)

## 1 · Loading the raw file
Same loader as in the audit (encoding and separator detected). The SHA-256 is saved in the report so that the exact file used can be identified later.

In [ ]:
raw, meta = prep.read_raw(RAW_PATH)
for k, v in meta.items():
    print(f"{k:10s}: {v}")
raw.head(3)

**What I see.** The file is the same as in the audit: 1 872 rows, 11 columns, encoding `utf-8-sig`, separator `;`, 21.49 MB. I keep its SHA-256 so that I can say later which version of the file I used.

## 2 · Column names
Accents, spaces and brackets in the column names make the code and the Streamlit widgets fragile. I convert everything to `snake_case` and give the six production columns a short name (`nuclear_gwh`, `thermal_gwh`, `hydro_gwh`, `wind_gwh`, `solar_gwh`, `bioenergy_gwh`). The unit is GWh per region and per month, as found in the audit.

In [ ]:
raw_std = prep.standardize_columns(raw)
mapping = pd.DataFrame({"original column": raw.columns, "new name": raw_std.columns})
mapping

## 3 · Dates and region labels
**Dates.** `mois` is text like `2025-12`. I parse it to a real date (first day of the month). The notebook stops if one value cannot be parsed.

**Regions.** The audit showed that code and name are one-to-one and that the 13 names are the post-2016 regions. To be sure that every later join uses the same label, I take the name from the INSEE code (official list in `config.py`) and keep the original label in `region_label_raw`. The INSEE code is stored as text, because it is an identifier and not a quantity.

In [ ]:
step1, info = prep.parse_dates_and_regions(raw_std)
print("date dtype            :", step1["date"].dtype)
print("period                :", step1["date"].min().date(), "->", step1["date"].max().date())
print("unparsed dates        :", info["unparsed_dates"])
print("labels changed        :", info["labels_changed"], "rows (original label different from the official name)")
print("region_code dtype     :", step1["region_code"].dtype, "| values:", sorted(step1["region_code"].unique()))

(step1.groupby(["region_code", "region", "region_label_raw"]).size().rename("rows").reset_index())

**What I see.** All the dates were parsed (0 failures) and the period is 2014-01 to 2025-12. The 13 names of the file already are the official ones, so no label had to be changed (0 rows). Each region has 144 rows, one per month. The INSEE code is now text.

## 4 · Structural nuclear missing values
In the audit, 864 nuclear values were missing and nothing else. 864 is exactly 6 regions × 144 months, so I suspected regions with no nuclear plant. Before replacing anything, I test it region by region: a structural case means *all* 144 months are empty; scattered gaps would mean real holes and the rule would not be valid.

In [ ]:
status = prep.nuclear_status(step1)
display(status.sort_values(["status", "region"]))

n_struct = int((status["status"] == "structural (never reported)").sum())
n_partial = int((status["status"] == "partial (real gaps)").sum())
print(f"regions never reporting nuclear : {n_struct}")
print(f"regions with partial gaps       : {n_partial}")
assert n_partial == 0, "partial gaps found: the structural rule cannot be applied as is"

Rule applied: for the regions where nuclear is empty in all months, `NaN` becomes `0 GWh`. No other value is imputed. Each modified row is flagged (`flag_nuclear_structural_zero`) and the region gets `has_nuclear = False`.

In [ ]:
step2 = prep.apply_structural_rule(step1, status)
before = int(step1[list(cfg.GWH.values())].isna().sum().sum())
after = int(step2[list(cfg.GWH.values())].isna().sum().sum())
print(f"missing values in the six measures: before = {before}, after = {after}")
print("rows flagged as structural zero   :", int(step2["flag_nuclear_structural_zero"].sum()))
print("regions without nuclear           :", sorted(step2.loc[~step2["has_nuclear"], "region"].unique()))

**What I see.** Exactly six regions never report nuclear: Bourgogne-Franche-Comté, Bretagne, Corse, Pays de la Loire, Provence-Alpes-Côte d'Azur and Île-de-France. The seven others (Auvergne-Rhône-Alpes, Centre-Val de Loire, Grand Est, Hauts-de-France, Normandie, Nouvelle-Aquitaine, Occitanie) have no missing month. No region has scattered gaps.

**What I conclude.** The 864 missing values are not holes in the data. They are all-or-nothing by region and they match the regions without a nuclear plant, so a missing value means "no production" and 0 GWh is the right value. After the rule the six measures have no missing value left (864 to 0). I do not impute anything else.

## 5 · Quality flags
I do not delete anything. I flag:
- negative values (the audit found 34: nuclear 6, thermal 26, wind 2),
- extreme values, compared with the **region's own history** (below Q1 − 3×IQR or above Q3 + 3×IQR). In the audit I used one global rule, which mostly flagged big regions; this version compares each region with itself,
- structural zeros and any remaining missing value.

`quality_flags` gives the list of flags of a row, or `ok`.

In [ ]:
step3 = prep.add_quality_flags(step2)

flag_counts = pd.Series({
    "ok": int((step3.quality_flags == "ok").sum()),
    "nuclear_structural_zero": int(step3.flag_nuclear_structural_zero.sum()),
    "negative_value": int(step3.flag_negative_value.sum()),
    "extreme_value": int(step3.flag_extreme_value.sum()),
    "missing_value": int(step3.flag_missing_value.sum()),
}, name="rows")
display(flag_counts.to_frame())

neg_rows = []
for s in cfg.SOURCES:
    col = cfg.GWH[s]
    r = step3.loc[step3[col] < 0, ["date", "region", col]].rename(columns={col: "value_gwh"})
    r.insert(2, "source", s)
    neg_rows.append(r)
neg_table = pd.concat(neg_rows).sort_values("value_gwh")
print(f"\nnegative values: {len(neg_table)}")
display(neg_table.head(15))

ext_by_source = {s: int(prep._extreme(step3[cfg.GWH[s]], step3["region_code"], cfg.EXTREME_IQR_K).sum()) for s in cfg.SOURCES}
print("extreme values by source:", ext_by_source)

print("\nnegative values by source and region:")
display(neg_table.groupby(["source", "region"]).agg(n=("value_gwh", "size"), first=("date", "min"), last=("date", "max"),
                                                    smallest_gwh=("value_gwh", "min")).round(2))
ext_solar = prep._extreme(step3["solar_gwh"], step3["region_code"], cfg.EXTREME_IQR_K)
print("extreme solar values by year:", step3.loc[ext_solar].groupby(step3.loc[ext_solar, "date"].dt.year).size().to_dict())
print("extreme solar values by region:", step3.loc[ext_solar].groupby("region").size().to_dict())

**What I see.** 965 rows have no flag, 864 rows are structural zeros, 34 values are negative and 39 values are extreme. All the extreme values are solar; there is none for the other sources.

**What I conclude.**
- The six negative nuclear values all belong to Occitanie, from April to September 2023 (between about -9 and -50 GWh). The most negative thermal values (about -2 to -4.5 GWh) are in Occitanie in 2014-2015 and in Pays de la Loire in 2020 and 2025. They are small compared with the monthly production of these regions. I read them as net values (the plant used more electricity than it produced during the month). The file does not explain them, so this is an assumption of mine.
- Solar is the only source with extreme values. Solar grew a lot over the period, so I looked at when they happen (count by year and by region printed above). I keep them and use the flag only to annotate charts; I will look at them again in the EDA.
- I do not delete any row. I keep the raw value and the flag in `cleaned_dataset`, and I set the negative values to 0 only in the feature dataset (section 7.0).

## 6 · Cleaned tables
The raw file weighs about 21 MB for 1 872 rows, because the region polygon is repeated on every row. I split the data into a small star schema:

- `cleaned_dataset` : region × month, wide format (one column per source),
- `fact_energy` : month × region × source (long format, easy to filter and group in the app),
- `dim_region` : one row per region, with the centroid and the validated polygon,
- `dim_source` : family and colour of each source.

The geometry is exported once as `regions.geojson`, so the map in the app needs no external file.

In [ ]:
clean, status, info = prep.clean_dataset(raw_std)

# the single function must give the same table as the step-by-step version above
stepwise = step3[prep.CLEAN_COLUMNS].sort_values(["date", "region_code"]).reset_index(drop=True)
pd.testing.assert_frame_equal(clean, stepwise)

dim_region, geojson = prep.build_dim_region(raw_std, clean)
dim_source = prep.build_dim_source()
fact = prep.build_fact(clean)

print("cleaned_dataset :", clean.shape)
print("fact_energy     :", fact.shape)
print("dim_region      :", dim_region.shape, "| valid geometry:", int(dim_region.geo_valid.sum()), "/ 13")
print("dim_source      :", dim_source.shape)
display(dim_region.drop(columns=["region_slug"]))
fact.head(8)

**What I see.** `cleaned_dataset` keeps the 1 872 rows (17 columns). `fact_energy` has 11 232 rows, which is 1 872 × 6 sources. `dim_region` has 13 rows and the 13 polygons are valid (all MultiPolygon, from 169 points for Corse to 617 for Nouvelle-Aquitaine); the centroids fall inside France. The polygon now exists once per region instead of 1 872 times, which was the cause of the 21 MB.

## 7 · Feature engineering
I build the features group by group, with the functions of `utils/prep.py`. Every feature is documented in section 8 (formula, meaning, use for the research question).

Three choices I make here and that I will state in the report:

1. **Negative values.** The values are net figures and 34 of them are negative. In `cleaned_dataset` and `fact_energy` I keep the raw value and the flag. In the feature dataset I set them to 0 (section 7.0), so that every share stays between 0 and 100 and the six shares add up to 100.
2. **Growth.** Month-on-month growth is very seasonal, so I only use it for the total production. The growth of renewables, solar and wind is the year-on-year change (same month, one year before), which removes the seasonality.
3. **12-month windows.** Solar peaks in summer, wind and hydro in winter. For levels, ranks and scores I use 12-month sums. In December, a 12-month sum is the calendar year.

### 7.0 Negative values in the feature dataset
My first validation run failed on the range of the shares (see the observation after 7.3): with the 34 negative values, some totals become too small and a share can go above 100 %. A negative production cannot be shown as such on a dashboard, so for the feature dataset I set the negative values to 0. The raw values stay in `cleaned_dataset` and in `fact_energy`, and `flag_negative_value` marks the rows. I print the size of the adjustment to show that it is small.

In [ ]:
feat = clean.merge(dim_region[["region_code", "region_short"]], on="region_code", how="left")
feat = feat.sort_values(["region_code", "date"]).reset_index(drop=True)      # needed for shift / rolling
feat = prep.floor_negatives(feat)

gcols = list(cfg.GWH.values())
adj_by_source = -clean[gcols].clip(upper=0).sum()
adj_total = float(adj_by_source.sum())
adj_pct = adj_total / float(clean[gcols].clip(lower=0).sum().sum()) * 100
print("cells set to 0  :", int((clean[gcols] < 0).sum().sum()))
print(f"energy adjusted : {adj_total:.1f} GWh = {adj_pct:.4f} % of the total production")
adj_by_source.rename("GWh set to 0").to_frame().T

### 7.1 Time features
`year`, `month`, `quarter`, `season` (winter = Dec–Feb, spring = Mar–May, summer = Jun–Aug, autumn = Sep–Nov).

In [ ]:
feat = prep.add_time_features(feat)
display(feat[["date", "year", "month", "quarter", "season"]].drop_duplicates("month").sort_values("month").head(12))
print(feat["season"].value_counts().to_string())

### 7.2 Production metrics
`total_gwh` (all sources), `renewable_gwh` (hydro + wind + solar + bioenergy), `new_renewables_gwh` (wind + solar + bioenergy, without hydro). Hydro is old and depends on rain, so `new_renewables_gwh` is the part of the transition that is really growing.

In [ ]:
feat = prep.add_production_metrics(feat)
feat[["total_gwh", "renewable_gwh", "new_renewables_gwh"]].describe().T

**What I see.** A region produces about 3 350 GWh per month on average (maximum about 12 900, minimum about 93). Renewables are about 770 GWh per month on average, and roughly half of it (about 375 GWh) is wind, solar and bioenergy. The regions are very different in size, which is why I compare shares and not GWh.

### 7.3 Shares
Every share is `source / total_gwh × 100`. `low_carbon_share_pct` adds nuclear to the renewables. Shares are empty only if a total is zero or negative.

In [ ]:
feat = prep.add_shares(feat)
share_cols = [c for c in feat.columns if c.endswith("_share_pct")]
display(feat[share_cols].describe().T[["count", "min", "mean", "max"]])
gap = (feat[[f"{s}_share_pct" for s in cfg.SOURCES]].sum(axis=1) - 100).abs().max()
print(f"largest gap between the sum of the six shares and 100: {gap:.2e}")

**What I see.** The six shares add up to 100 on every row. In my first run, before section 7.0, the minimum and maximum were not between 0 and 100: the renewable share reached 104 % and the nuclear share went down to -4.3 %. The cause was the negative values, which make the total smaller than the renewable production in a few region-months. This is what made the range check fail. With the negative values set to 0 (section 7.0) the shares stay between 0 and 100.

### 7.4 Growth
`monthly_growth_pct` and `yearly_growth_pct` are computed on the total production. `renewable_growth_pct`, `new_renewables_growth_pct`, `solar_growth_pct` and `wind_growth_pct` are year-on-year. The growth is left empty when the base value is 0 or negative. Early values of solar and wind are small, so some growth rates are very large: I look at the distribution before using these columns.

In [ ]:
feat = prep.add_growth(feat)
growth_cols = ["monthly_growth_pct", "yearly_growth_pct", "renewable_growth_pct",
               "new_renewables_growth_pct", "solar_growth_pct", "wind_growth_pct"]
q = feat[growth_cols].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).T
q["empty"] = feat[growth_cols].isna().sum()
q["empty %"] = (feat[growth_cols].isna().mean() * 100).round(1)
q.insert(0, "median", feat[growth_cols].median())
growth_med = feat[growth_cols].median().round(1)
q

**What I see.** The median year-on-year growth is about +1 % for the total production, +6.5 % for renewables, +10 % for wind and for wind + solar + bioenergy together, and +15 % for solar. The spread is large: wind goes from about -62 % (1st percentile) to +239 % (99th percentile), solar from about -38 % to +123 %. The monthly growth is empty 13 times (the first month of each region). The year-on-year growth is empty 156 times (12 months × 13 regions), and 158 times for wind because two base values are not strictly positive.

**What I conclude.** Solar has the highest median growth, but wind is much more volatile from one year to the next, which is probably the weather. The monthly growth rates are noisy, so for the comparisons between regions I will use the 12-month features of the next section.

### 7.5 Rolling 12 months
`rolling_12m_renewable`, `rolling_12m_solar`, `rolling_12m_wind` are sums over the last 12 months of each region (empty during the first 11 months). I also keep the 12-month total, the 12-month shares, and the gain of the share over 12 months (in percentage points). These smooth columns are the basis of the ranks and of the scores.

In [ ]:
feat = prep.add_rolling(feat)
roll_cols = [c for c in feat.columns if "12m" in c]
display(feat[roll_cols].describe().T[["count", "min", "mean", "max"]])
print("empty values per region, rolling_12m_renewable        :", [int(x) for x in sorted(feat.groupby("region_code")["rolling_12m_renewable"].apply(lambda s: int(s.isna().sum())).unique())])
print("empty values per region, renewable_share_gain_12m_pts :", [int(x) for x in sorted(feat.groupby("region_code")["renewable_share_gain_12m_pts"].apply(lambda s: int(s.isna().sum())).unique())])

**What I see.** Each region has 11 empty months in the 12-month sums and 23 in the gains, as expected (1 729 and 1 573 non-empty rows). The 12-month shares are very different between regions: renewable share from about 2 % to 93 %, wind from 0.4 % to 59 %, solar from 0.1 % to 30 %, and thermal share up to 76 %. The largest 12-month totals are about 124 000 GWh for all sources and about 38 900 GWh for renewables.

### 7.6 Rankings
Rank of the region among the 13 regions **in the same month**, 1 = best, computed on 12-month values:
- `region_rank_renewables` : renewable share (12 months),
- `region_rank_solar` : solar share (12 months),
- `region_rank_wind` : wind share (12 months),
- `region_rank_growth` : gain of the renewable share over the last 12 months,
- `region_rank_renewables_peer` : same as the first one, but only among the regions of the same group (with nuclear / without nuclear). I added it after looking at the ranking below.

I rank on shares and not on GWh, to compare the intensity of the transition and not the size of the region.

In [ ]:
feat = prep.add_rankings(feat)
rank_cols = ["region_rank_renewables", "region_rank_renewables_peer", "region_rank_solar", "region_rank_wind", "region_rank_growth"]
last_month = feat["date"].max()
last_df = feat[feat["date"] == last_month].set_index("region")
display(last_df[["has_nuclear", *rank_cols]].sort_values("region_rank_renewables"))

top_level = ", ".join(last_df.sort_values("region_rank_renewables").index[:3])
top_growth = ", ".join(last_df.sort_values("region_rank_growth").index[:3])
bottom_level = ", ".join(last_df.sort_values("region_rank_renewables").index[-3:])
bottom_growth = ", ".join(last_df.sort_values("region_rank_growth").index[-3:])
print(f"top 3 on level  : {top_level}\ntop 3 on growth : {top_growth}")
print(f"last 3 on level : {bottom_level}\nlast 3 on growth: {bottom_growth}")

grp_cols = ["renewable_share_12m_pct", "new_renewables_share_12m_pct", "thermal_share_12m_pct",
            "renewable_share_gain_12m_pts"]
grp_tbl = last_df.groupby("has_nuclear")[grp_cols].mean().round(1)
grp_tbl.insert(0, "regions", last_df.groupby("has_nuclear").size())
print("\nmean by group in the last month (has_nuclear False / True):")
grp_tbl

**What I see.** The top of the level ranking (`region_rank_renewables`) is held by regions that have no nuclear plant (Bourgogne-Franche-Comté, Provence-Alpes-Côte d'Azur, Pays de la Loire, Bretagne, Île-de-France), and the last places are regions with nuclear. The top of the growth ranking is not the same list of regions, so the two rankings do not tell the same story.

**What I conclude.** A high renewable share in a region without nuclear partly means that the region produces little of anything else, it does not mean that it built a lot of renewables. This is why I added `region_rank_renewables_peer`, which ranks a region only against the regions of its own group. For the story I will not present the level ranking alone: I will always put it next to the progress (`region_rank_growth`) and say that the starting point is not the same for the two groups.

### 7.7 Storytelling features
These four features summarise the story in a form I can put on a map or in a ranking.

- **`mix_diversity_index`** (0–1): normalised Simpson index of the six shares, `(1 − Σ share²) / (1 − 1/6)`. 0 = one source only, 1 = six equal sources.
- **`fossil_dependency_score`** (0–100): 12-month thermal share, scaled between the 1st and 99th percentile of the dataset.
- **`energy_transition_score`** (0–100): weighted sum of four 0–100 components (renewable share 35 %, wind + solar + bioenergy share 30 %, low fossil dependency 20 %, diversity 15 %). The weights are my assumption, written in `config.py`. The components are stored so the app can change the weights (what-if).
- **`renewable_momentum_score`** (0–100): speed of the last 12 months. It uses the gain of the renewable share and of the wind + solar + bioenergy share, centred so that 50 = no change and above 50 = progress.

The scaling bounds come from the data (percentiles) and are saved in `feature_scales.json`. The scores are empty during the first 11 months (12-month window); the momentum is empty during the first 23 months.

In [ ]:
feat = prep.add_diversity(feat)
scales = prep.fit_scales(feat)
feat = prep.add_scores(feat, scales)
print(json.dumps({k: [round(x, 3) for x in v] if isinstance(v, list) else round(v, 3) for k, v in scales.items()}, indent=1))

score_cols = ["mix_diversity_index", "mix_diversity_12m", "fossil_dependency_score", "energy_transition_score",
              "renewable_momentum_score"]
feat[score_cols].describe().T

**What I see.** The scores stay inside 0-100 and the diversity index inside 0-1. The scaling bounds printed above come from the 1st and 99th percentiles of the dataset, so a few values are capped at 0 or 100. The 12-month scores have 1 729 values (the first 11 months of each region are empty) and the momentum has fewer, because it needs two years.

**What I conclude.** The scores are relative: they compare regions with each other inside this file and they depend on my weights. I will say it in the app and I will test the weights in section 10.

In [ ]:
# the step-by-step table must be identical to the one function used by the app
feat = feat.drop(columns=["region_label_raw"], errors="ignore").sort_values(["date", "region_code"]).reset_index(drop=True)
feat_check, scales_check = prep.build_features(clean, dim_region)
pd.testing.assert_frame_equal(feat, feat_check, check_dtype=False)
assert scales == scales_check

order = [c for c in dictionary.feature_order() if c in feat.columns]
assert set(order) == set(feat.columns), set(feat.columns) ^ set(order)
feat = feat[order]
print("feature dataset:", feat.shape)

**What I see.** The step-by-step table is identical to the one returned by `prep.build_features` (the assertion passes), and all the columns are documented. The notebook and the app will therefore give the same numbers.

## 8 · Feature dictionary
For every column: group, unit, formula, business meaning, and why it helps to answer the research question. The same table is exported as `feature_dictionary.csv` and copied in the feature report.

In [ ]:
fdict = dictionary.feature_dictionary()
print("documented columns:", len(fdict))
display(fdict.groupby("group", sort=False).size().rename("columns").to_frame())
fdict[fdict.group.isin(["Production metrics", "Advanced"])][["feature", "formula", "business_meaning", "research_value"]]

**What I see.** Every column of the feature dataset has a group, a unit, a formula, a business meaning and a reason why it helps the research question. I use this table as the reference when I build the pages of the app.

## 9 · Pre-aggregated tables for the app
To keep the dashboard fast, I prepare two small tables:
- `annual_region_summary` : region × year (annual sums and shares, December values of the 12-month features, ranks and the change since 2014),
- `national_annual` : France (13 regions) × year.

I do not read them here; I only check that they are consistent (section 11).

In [ ]:
annual = prep.build_annual(feat)
national = prep.build_national(annual)
print("annual_region_summary:", annual.shape, "| national_annual:", national.shape)
annual.head(3)

**What I see.** `annual_region_summary` has one row per region and year (13 × 12 = 156 rows) and `national_annual` has one row per year (12 rows). The ranks and scores in the annual table are the December values, so they cover the whole calendar year.

**Point to remember for the app.** The national share has to be computed from the sums of GWh (weighted), not by averaging the 13 regional shares. The mean of the regional shares is higher (about 39.8 % on 12 months) than the national share on the whole period (about 23.0 %), because small regions without nuclear count as much as large ones in a simple mean. In the dashboard I will always write which of the two I show.

## 10 · Robustness of the transition score
The weights of `energy_transition_score` are an assumption. To check that the ranking of the regions does not depend on them, I recompute the score of the latest month with four weight schemes and compare the rankings (Spearman correlation).

In [ ]:
schemes = {
    "default": cfg.WEIGHTS_TRANSITION,
    "equal": {"renewable": .25, "new_renewables": .25, "fossil_free": .25, "diversity": .25},
    "new renewables first": {"renewable": .20, "new_renewables": .50, "fossil_free": .20, "diversity": .10},
    "no diversity": {"renewable": .40, "new_renewables": .35, "fossil_free": .25, "diversity": 0.0},
}
last = feat[feat["date"] == feat["date"].max()].set_index("region")
sens = pd.DataFrame({k: prep.weighted_transition_score(last, w) for k, w in schemes.items()})
rank_tbl = sens.rank(ascending=False, method="min").astype(int)
display(rank_tbl.sort_values("default"))
spearman = sens.corr(method="spearman")
display(spearman.round(3))
print("lowest Spearman correlation between two schemes:", round(float(spearman.values.min()), 3))
max_move = int(rank_tbl.sub(rank_tbl["default"], axis=0).abs().to_numpy().max())
print("largest change of rank of a region between the default and another scheme:", max_move)

**What I see.** The Spearman correlations between the default ranking and the three other schemes are between 0.967 and 0.989, and no region moves by more than 2 places.

**What I conclude.** The ranking of the regions does not depend much on my weights. The extreme positions (best and worst) stay the same, only regions in the middle swap places. I can therefore use the default weights in the story and keep the sliders of the app as an exploration tool.

## 11 · Validation checks
Each check is PASS or FAIL. They cover the cleaned data (keys, dates, totals unchanged against the raw file and against notebook 01), the star schema, the features (identities between columns, shares adding up to 100, empty values of the rolling windows, ranges of the scores and ranks) and the documentation (every column described).

In [ ]:
checks = prep.run_checks(raw_std, clean, fact, dim_region, feat, annual, fdict)
n_pass, n_fail = int((checks.status == "PASS").sum()), int((checks.status == "FAIL").sum())
display(checks)
print(f"\n{n_pass} PASS / {n_fail} FAIL out of {len(checks)} checks")
if n_fail:
    print("FAILED:", checks.loc[checks.status == "FAIL", "check"].tolist())

**What I see.** In my first run one check failed: the range of the monthly shares (see 7.3). I fixed it in 7.0 (negative values set to 0 in the feature dataset only) and I added three checks: shares inside [0, 100], feature values equal to the cleaned values except the negative cells, and peer ranks inside the group size. The count printed above is the one I keep. The two checks that compare the totals with notebook 01 confirm that cleaning did not change any GWh total of the raw file.

**What I conclude.** The datasets are consistent: keys are unique, totals are unchanged by the cleaning, the star schema gives back the wide table, and the identities between features hold. I can start the EDA from these files.

## 12 · Exports
CSV files are UTF-8 with a comma separator. Derived features are rounded to 6 decimals; dates are ISO (`YYYY-MM-DD`). If `pyarrow` is installed, I also save a parquet copy of the feature table (faster to load in Streamlit).

In [ ]:
def rounded(df, nd=6):
    out = df.copy()
    fl = out.select_dtypes(include=["float"]).columns
    out[fl] = out[fl].round(nd)
    return out

files = {
    "cleaned_dataset.csv": clean,
    "feature_engineered_dataset.csv": feat,
    "dim_region.csv": dim_region,
    "fact_energy.csv": fact,
    "dim_source.csv": dim_source,
    "annual_region_summary.csv": annual,
    "national_annual.csv": national,
    "feature_dictionary.csv": fdict,
}
for name, d in files.items():
    rounded(d).to_csv(PROC / name, index=False, encoding="utf-8", date_format="%Y-%m-%d")

(PROC / "regions.geojson").write_text(json.dumps(geojson), encoding="utf-8")
(PROC / "feature_scales.json").write_text(json.dumps({"scales": scales, "weights_transition": cfg.WEIGHTS_TRANSITION,
                                                      "weights_momentum": cfg.WEIGHTS_MOMENTUM,
                                                      "percentiles": cfg.SCALE_PERCENTILES}, indent=2), encoding="utf-8")
try:
    rounded(feat).to_parquet(PROC / "feature_engineered_dataset.parquet", index=False)
except Exception as e:
    print("parquet not written:", type(e).__name__)

sizes = pd.DataFrame({"file": [p.name for p in sorted(PROC.iterdir())],
                      "size_MB": [round(p.stat().st_size / 1e6, 3) for p in sorted(PROC.iterdir())]})
sizes.loc[len(sizes)] = ["(raw file, for comparison)", meta["size_mb"]]
sizes

**What I see.** The raw file is 21.49 MB because the polygon of the region is repeated on each of the 1 872 rows. The cleaned and feature datasets are below 1 MB each, and the polygons are kept once in `regions.geojson`. This will make the app load much faster.

In [ ]:
# reload what was written: the files must give back the same numbers
back = pd.read_csv(PROC / "feature_engineered_dataset.csv", parse_dates=["date"], dtype={"region_code": str})
print("reloaded shape:", back.shape, "| same columns:", list(back.columns) == list(feat.columns))
print("renewable_gwh total (written vs memory):", round(back.renewable_gwh.sum(), 3), "/", round(feat.renewable_gwh.sum(), 3))

**What I see.** The file I wrote and read back has the same columns and the same renewable total as the table in memory, so the export does not lose anything (6 decimals only).

## 13 · Reports
The two markdown reports are written by the notebook from the results above, so the numbers in the reports are always the ones of the last run.

In [ ]:
today = pd.Timestamp.today().date()
sev = prep.to_markdown_table
ren_cols = [cfg.GWH[x] for x in cfg.RENEWABLE]
overview_hydro = clean['hydro_gwh'].sum() / clean[ren_cols].sum().sum() * 100

neg_by_source = neg_table.groupby("source").size().reindex(cfg.SOURCES).dropna().astype(int)
steps = pd.DataFrame([
    ["1", "Read the file", f"encoding {meta['encoding']}, separator '{meta['separator']}'", f"{meta['rows']:,} rows, {meta['columns']} columns"],
    ["2", "Column names", "snake_case, short names for the six sources", "11 columns renamed"],
    ["3", "Dates", "'YYYY-MM' text parsed to the first day of the month", f"{info['unparsed_dates']} unparsed"],
    ["4", "Regions", "name taken from the INSEE code, code kept as text", f"{info['labels_changed']} labels changed"],
    ["5", "Structural nuclear", "NaN to 0 where the region never reports nuclear", f"{int(clean.flag_nuclear_structural_zero.sum())} cells, {n_struct} regions"],
    ["6", "Quality flags", "negative, extreme (own region, 3 x IQR), structural, missing", f"{int((clean.quality_flags != 'ok').sum())} rows with at least one flag"],
    ["7", "Star schema", "fact_energy, dim_region, dim_source", f"{len(fact):,} fact rows"],
], columns=["#", "step", "rule", "result"])

flag_tbl = flag_counts.rename_axis("flag").reset_index()
flag_tbl["share of rows %"] = (flag_tbl["rows"] / len(clean) * 100).round(2)
nuc_tbl = status.reset_index()[["region_code", "region", "missing", "rows", "status"]]
chk_tbl = checks[["layer", "check", "status", "observed"]]

dq_report = f"""# Data quality report

Project: Renewable energy transition in France - regional dynamics
Author: Soumahoro Souleymane
Produced by `notebooks/02_cleaning_feature_engineering.ipynb` on {today}

## 1. Source file

| item | value |
|---|---|
| file | `{meta['file']}` |
| size | {meta['size_mb']} MB |
| rows x columns | {meta['rows']:,} x {meta['columns']} |
| encoding / separator | {meta['encoding']} / `{meta['separator']}` |
| SHA-256 | `{meta['sha256']}` |
| period | {clean.date.min():%Y-%m} to {clean.date.max():%Y-%m} ({clean.date.nunique()} months) |
| regions | {clean.region_code.nunique()} metropolitan regions |
| unit | GWh per region per month |
| portal | data.gouv.fr / ODRE (licence and citation: see README) |

## 2. Cleaning steps

{sev(steps)}

## 3. Structural nuclear values

The audit reported {int(raw_std.nuclear_gwh.isna().sum())} empty nuclear values. I tested them region by region: a region is structural when all of its {clean.date.nunique()} months are empty. Result: {n_struct} regions never report nuclear and {n_partial} region has scattered gaps. For the {n_struct} regions the empty value was replaced by 0 GWh and flagged (`flag_nuclear_structural_zero`, `has_nuclear = False`). No other value was imputed.

{sev(nuc_tbl)}

## 4. Quality flags

{sev(flag_tbl)}

Negative values by source: {', '.join(f'{k} = {v}' for k, v in neg_by_source.items())}. The smallest value is {neg_table.value_gwh.min():.3f} GWh. Values are net production, so I do not remove them: the raw value and the flag stay in `cleaned_dataset` and `fact_energy`. Only in the feature dataset the {int((clean[gcols] < 0).sum().sum())} negative cells are set to 0 ({adj_total:.1f} GWh, {adj_pct:.4f} % of the production), so that every share stays between 0 and 100.

Extreme values (compared with the region's own history, 3 x IQR) by source: {', '.join(f'{k} = {v}' for k, v in ext_by_source.items())}. They are used to annotate charts, not to delete rows.

## 5. Validation checks

Result: **{n_pass} PASS / {n_fail} FAIL** out of {len(checks)} checks.

{sev(chk_tbl)}

## 6. Known limitations

1. The file contains production only, not consumption: shares are shares of regional production.
2. Six sources and metropolitan France only (no overseas regions); no installed capacity, price or CO2 data.
3. Hydro is the largest renewable source and depends on rainfall; the growth of wind, solar and bioenergy is the clearer transition signal.
4. The renewable / nuclear / thermal classification is an assumption defined in `utils/config.py`.
5. Growth rates are noisy when the base month is small (early solar and wind values); 12-month features are used for comparisons.
6. Licence and exact definitions of the columns must be confirmed on the portal page.

## 7. Reproducibility

Run `notebooks/01_dataset_audit.ipynb`, then `notebooks/02_cleaning_feature_engineering.ipynb`. All rules are in `utils/prep.py` and `utils/config.py`. The SHA-256 above identifies the input file.
"""
(REPORTS / "data_quality_report.md").write_text(dq_report, encoding="utf-8")
print(len(dq_report.splitlines()), "lines written to", REPORTS / "data_quality_report.md")

In [ ]:
groups = fdict.groupby("group", sort=False)["feature"].apply(list)
group_tbl = pd.DataFrame({"group": groups.index, "columns": [len(v) for v in groups.values],
                          "features": [", ".join(f"`{x}`" for x in v) if len(v) <= 12 else ", ".join(f"`{x}`" for x in v[:12]) + ", ..."
                                       for v in groups.values]})
weights_tbl = pd.DataFrame({"component": list(cfg.WEIGHTS_TRANSITION), "weight": list(cfg.WEIGHTS_TRANSITION.values())})
mom_tbl = pd.DataFrame({"component": list(cfg.WEIGHTS_MOMENTUM), "weight": list(cfg.WEIGHTS_MOMENTUM.values())})
scale_tbl = pd.DataFrame([[k, round(v[0], 3), round(v[1], 3)] if isinstance(v, list) else [k, -round(v, 3), round(v, 3)]
                          for k, v in scales.items()], columns=["component", "lower bound", "upper bound"])
rank_md = rank_tbl.sort_values("default").reset_index()
fdict_md = fdict[["feature", "group", "unit", "formula", "business_meaning", "research_value"]]
n_ok = int((checks[checks.layer == "features"].status == "PASS").sum())
n_all = int((checks.layer == "features").sum())

fe_report = f"""# Feature engineering report

Project: Renewable energy transition in France - regional dynamics
Author: Soumahoro Souleymane
Produced by `notebooks/02_cleaning_feature_engineering.ipynb` on {today}

## 1. Objective

The research question is *how are French regions progressing toward a renewable electricity future?* The raw file only gives GWh by source, region and month. I need measures of **level** (how renewable is a region), **progress** (how fast it changes), **position** (rank among regions) and **balance** (diversity and fossil dependence). This report documents the features I built for that purpose.

## 2. Pipeline

```
data/production-regionale-mensuelle-filiere.csv
   -> cleaned_dataset.csv ({clean.shape[0]:,} x {clean.shape[1]})
   -> feature_engineered_dataset.csv ({feat.shape[0]:,} x {feat.shape[1]})
   -> fact_energy.csv ({fact.shape[0]:,} x {fact.shape[1]}), dim_region.csv, dim_source.csv
   -> annual_region_summary.csv ({annual.shape[0]} x {annual.shape[1]}), national_annual.csv ({national.shape[0]} x {national.shape[1]})
```

Grain of the datasets: one row = one region and one month ({clean.region_code.nunique()} regions x {clean.date.nunique()} months), except `fact_energy` (x 6 sources) and the annual tables.

## 3. Feature groups

{sev(group_tbl)}

## 4. Design decisions

1. **Negative values.** Net production gives {int((clean[gcols] < 0).sum().sum())} negative cells. They are kept (with a flag) in the cleaned and fact tables and set to 0 in the feature dataset, so that the six shares stay between 0 and 100 and add up to 100. The adjustment is {adj_pct:.4f} % of the production.
2. **Renewable vs new renewables.** Hydro is {overview_hydro:.0f}% of the renewable GWh in this file (see notebook 01). `new_renewables` (wind + solar + bioenergy) isolates the part of the transition that is growing.
3. **Growth.** Month-on-month growth is seasonal, so it is computed on total production only and kept for information. Growth of renewables, solar and wind is year-on-year (same month one year before). It is empty when the base is 0 or negative and can be very large when the base is small.
4. **12-month windows.** Solar peaks in summer and wind and hydro in winter and spring. Levels, ranks and scores use 12-month sums (in December this equals the calendar year, checked against a group-by).
5. **Ranks on shares.** Rankings compare the intensity of the transition and not the size of the region. 1 = best among the 13 regions in the same month; ties share the lowest rank. `region_rank_renewables_peer` ranks a region only inside its group (with / without nuclear), because the level ranking is dominated by the regions without nuclear.
6. **Structural nuclear zero.** Regions without nuclear get `has_nuclear = False`, so the dashboard can explain their different starting point.
7. **Same code for notebook and app.** All formulas are in `utils/prep.py`; the notebook asserts that the step-by-step result equals the function used by the app.

## 5. Scores

Both scores use the 12-month features, so they are empty for the first months of 2014 (11 months for the transition score, 23 for the momentum).

**Energy transition score (0-100)** = weighted sum of four components, each scaled between the 1st and the 99th percentile of the dataset (values outside are capped).

{sev(weights_tbl)}

**Renewable momentum score (0-100)** = weighted mean of the 12-month gain of the renewable share and of the wind + solar + bioenergy share, centred on 50 (50 = no change, above = progress).

{sev(mom_tbl)}

Scaling bounds (saved in `feature_scales.json`):

{sev(scale_tbl)}

**Robustness.** I recomputed the score of {feat.date.max():%Y-%m} with four weight schemes (default, equal weights, new renewables first, no diversity). The lowest Spearman correlation between two rankings is {float(spearman.values.min()):.3f}.

{sev(rank_md)}

The weights are an assumption of mine. The components are stored in the dataset so the app can offer a what-if slider on the weights.

## 6. Feature dictionary

{sev(fdict_md)}

## 7. Validation

{n_ok} of {n_all} feature checks pass (full table in `data_quality_report.md`). Every column of the exported datasets has an entry in the dictionary.

## 8. Limits of the features

- Shares measure regional **production**, not the share of local consumption.
- Scores depend on my weights and on percentile scaling: they rank regions inside this dataset, they are not absolute measures.
- Year-on-year growth of solar and wind is unstable at the beginning of the period (small base).
- Hydro varies with rainfall, which can move the renewable share from one year to another without any new capacity.
- The level of the renewable share mostly separates regions with and without nuclear; the progress features (gain of share, momentum) are the fairer comparison.
- Mean of the 13 regional shares and national share (weighted by GWh) are two different numbers; the app must say which one it shows.
- Robustness: the largest rank change of a region between weight schemes is {max_move} place(s) in {feat.date.max():%Y-%m}.
"""
(REPORTS / "feature_engineering_report.md").write_text(fe_report, encoding="utf-8")
print(len(fe_report.splitlines()), "lines written to", REPORTS / "feature_engineering_report.md")

## 14 · Summary of the run
This block lists what I need to read the results and write the analysis in the final version.

In [ ]:
print("=" * 78)
print("RUN SUMMARY")
print("=" * 78)
print(f"raw file          : {meta['file']}  ({meta['size_mb']} MB)  sha256 {meta['sha256'][:16]}...")
print(f"cleaned_dataset   : {clean.shape}   fact_energy: {fact.shape}   dim_region: {dim_region.shape}")
print(f"feature dataset   : {feat.shape}   annual: {annual.shape}   national: {national.shape}")
print(f"structural nuclear: {n_struct} regions -> {', '.join(sorted(dim_region.loc[~dim_region.has_nuclear, 'region']))}")
print(f"flags             : {flag_counts.to_dict()}")
print(f"negative values   : {neg_by_source.to_dict()}")
print(f"extreme values    : {ext_by_source}")
print(f"negatives -> 0    : {int((clean[gcols] < 0).sum().sum())} cells, {adj_total:.1f} GWh ({adj_pct:.4f} % of production)")
print(f"ranks last month  : top level = {top_level} | top growth = {top_growth}")
print(f"growth medians %  : {growth_med.to_dict()}")
print(f"checks            : {n_pass} PASS / {n_fail} FAIL of {len(checks)}")
print(f"score robustness  : lowest Spearman = {float(spearman.values.min()):.3f}")
print()
print("National renewable share by year (%), for the final commentary:")
print(national[["year", "total_gwh", "renewable_share_pct", "new_renewables_share_pct", "nuclear_share_pct", "thermal_share_pct"]].round(2).to_string(index=False))

**What I will do with this block.** I read the national shares by year, the checks table and the last-month ranks, and I use them as the first story line of the EDA (level, progress, then the difference between regions with and without nuclear).